# 122 — Evaluación y depuración de agentes

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

Dos ejes complementarios:

- **Eval de resultado:** ¿el estado final satisface el objetivo? (predicado
  ejecutable: tests pasan, archivo válido). Es lo que importa al usuario; ignora el
  camino.
- **Eval de proceso:** ¿el CÓMO fue correcto? — tools pertinentes, argumentos
  válidos, permisos respetados, presupuesto razonable. Detecta éxitos por casualidad
  (✓ resultado, ✗ proceso: fallará pronto) y fallos por una decisión reparable.

La matriz 2×2 resultado×proceso es la primera herramienta diagnóstica. Métricas sobre
un conjunto de tareas reproducible: tasa de éxito (con su varianza), pass@k, costo
por éxito, pasos vs óptimo, violaciones. LLM-as-judge para criterios blandos — con
rúbrica y calibración contra humanos.

### 🔬 Depurar = leer trayectorias

Método: (1) reunir trayectorias FALLIDAS del eval; (2) localizar el **primer paso
divergente** (la primera decisión que un experto no tomaría — el fallo visible suele
ser síntoma posterior); (3) clasificar la causa raíz (E1 instrucciones, E2 selección
de tool, E3 argumentos, E4 interpretación de la observación, E5 planificación,
E6 parada, E7 entorno/tool); (4) CONTAR y arreglar la categoría dominante;
(5) re-ejecutar el eval completo — sin re-ejecución no hay evidencia de mejora ni
detección de regresiones.

El laboratorio `evaluation` entrega la matriz mínima: tp=3, fp=1, fn=1 →
precision = recall = 0,75, con la advertencia honesta de que 8 ejemplos no estiman
desempeño real.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("evaluation", seed=122)
show(result)


## Reflexión

1. Un agente "arregló" el build borrando el test que fallaba: resultado ✓, proceso ✗.
   ¿Qué combinación de eval de proceso (122) y permisos (119) convierte ese caso en
   fallo visible, y por qué la tasa de éxito ingenua lo premiaba?
2. ¿Por qué el "primer paso divergente" es mejor unidad de diagnóstico que el paso
   donde el agente se rindió?
3. El laboratorio declara "ocho ejemplos no estiman desempeño real". ¿Qué decisión
   podrías tomar igualmente con esos 8 ejemplos y cuál sería irresponsable tomar?